<a href="https://colab.research.google.com/github/josedanielisidororeyes/Advanced_Data_Engineering/blob/main/Transformaciones_en_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matería: Ingeniería de Datos Avanzada:
# Alumno: Jose Daniel Isidoro Reyes
# Nombre de la tarea: Actividad Práctica: Transformaciones en PySpark
# Fecha: 17 de mayo de 2026.

In [2]:
# Carga de librerías
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [4]:
# Creación de sesión  en PySpark
spark =  (
    SparkSession.builder

    # Nombre de la aplicación
    .appName("Transformaciones_en_PySpark")

    # Desactivación de barra de progreso
    .config("spark.ui.showConsoleProgress", "false")

    # Creación de sesión o recuperación en caso de que exista
    .getOrCreate()
)

# Reducción de cantidad de mensajes del log
spark.sparkContext.setLogLevel('ERROR')

print(f"SparkSession activa - versión Spark: {spark.version}")

SparkSession activa - versión Spark: 4.0.2


Se realiza la conexión con PySpark a través de SparkSesion.

#1) Carga del dataset

In [40]:
ruta =   '/content/drive/MyDrive/Ingeniería de Datos Avanzada/NSL_KDD.csv'

# Carga del archivo csv en un DataFrame de PySpark
df_raw =  (
    spark.read

    # Instrucción de que el archivo  no  contiene encabezados
    .option('header', 'True')

    # Aplicar esquema automático
    .option("inferSchema", "true")
    .option("sep", ",")

    # Carga del archivo csv
    .csv(ruta)
)



Cabe hacer la aclaración que se detectó un desfase  en las columnas en la parte de filtrado de registros,  el cual fue corregido de manera manual en el archivo de origen recorriendo las columnas una posición hacia la derecha y fijando duration como primera columna. De igual modo, se realizó el cambio de nombre la columna class por label para estar homologado con los ejercicios.

# 2) Información general del DataFrame

In [41]:
# Impresión de número total de filas y columnas
print(f' Dataset cargado: {df_raw.count():,} filas')
print(f"Número de columnas: {len(df_raw.columns)}")


 Dataset cargado: 18,035 filas
Número de columnas: 42


In [42]:
# Impresión de las primeras 10 filas
df_raw.show(10, truncate =  True )

+-------------+-------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+------------+--------+
|protocol_type|service|flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|rerror_rate|srv_rerror_rate|same_srv_rate|diff_srv_rate|srv_diff_host_rate|dst_host_co

Se realiza una inspección del número de filas  y columnas utilizando count() y len(), respectivamente. De igual modo, se muestran las primeras 10 filas utilizando la acción .show()

# 3) Selección de columnas

In [43]:
df_raw.select(
    'protocol_type',
    'service',
    'src_bytes',
    'dst_bytes',
    'label'
).show()

+-------------+-------+---------+---------+------------+
|protocol_type|service|src_bytes|dst_bytes|       label|
+-------------+-------+---------+---------+------------+
|          tcp|private|        0|        0|     neptune|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|       15|       mscan|
|          tcp|   http|      267|    14515|      normal|
|          tcp| telnet|      129|      174|guess_passwd|
|          tcp|   http|      327|      467|      normal|
|          tcp|    ftp|       26|      157|guess_passwd|
|          tcp| telnet|        0|        0|       mscan|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|        0|     neptune|
|          tcp| telnet|      773|   364200|      normal|
|          tcp|   http|      350|     3610|      normal|
|          tcp|   http|      246|     2090|      normal|
|          udp|private|       45|       44|      normal|
|          tcp|private|        

Se realiza una selección de 5 columnas realizando una tranformación del conjunto de datos utilizando .select() y a través de show mostramos los primeros registros.

# 4) Filtrado de registros

In [44]:
df_raw.filter(
    (F.col('protocol_type') == 'tcp') &
    (F.col('src_bytes') > 1000)
).show(10, truncate = True)

+-------------+--------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+-----------+--------+
|protocol_type| service|flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|rerror_rate|srv_rerror_rate|same_srv_rate|diff_srv_rate|srv_diff_host_rate|dst_host_c

Se realiza  otra transformación para filtrar los registros protocol_typ cuando son iguales a tcp y src_bytes mayores a 1000. A través de F. apuntamos directamente a las columnas necesarias para la comparación.

# 5) Creación de columnas

In [48]:
df_total = df_raw.withColumn(
    "total_bytes",
    F.col("src_bytes") + F.col("dst_bytes")
)

df_total.show(10, truncate  =  False)

+-------------+-------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+------------+--------+-----------+
|protocol_type|service|flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|rerror_rate|srv_rerror_rate|same_srv_rate|diff_srv_rate|srv_diff_host_rate

Posteriormente, se crea la columna total_bytes utilizando .withColumn().

# Ordenación de registros

In [50]:
# Orden de mayor a menor por total_bytes
df_total.orderBy(F.col('total_bytes').desc()).show(10, truncate =  False)

+-------------+--------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+------+--------+-----------+
|protocol_type|service |flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|rerror_rate|srv_rerror_rate|same_srv_rate|diff_srv_rate|srv_diff_host_rate|dst

En seguida, se realiza una ordenación del cojunto de datos  a través de .OrderBy(), el cual funciona de manera similar a ORDER BY en SQL.

# 7) Valores únicos

In [51]:
# Valores únicos de protocol_type
df_total.select('protocol_type').distinct().show(truncate =  False)

+-------------+
|protocol_type|
+-------------+
|tcp          |
|udp          |
|icmp         |
+-------------+



Se realiza  otra transformación, seleccionando protocol_type a través de .select() y con el uso de .distinct() se obtiene los registros únicos de la columna.

# 8) Agrupación de registros por conteo

In [52]:
# Agrupación por protocol_type
df_total.groupBy('protocol_type').count().show(10)


+-------------+-----+
|protocol_type|count|
+-------------+-----+
|          tcp|15136|
|          udp| 2074|
|         icmp|  825|
+-------------+-----+



Se realiza otra transformación del conjunto de datos  a través de groupBy() y posteriormente se hace una agregación utilizando .count()

# 9) Agrupación de registros por promedio

In [54]:
df_total.groupBy("label").agg(
    F.avg("src_bytes").alias("promedio_src_bytes")
).show()

+---------------+------------------+
|          label|promedio_src_bytes|
+---------------+------------------+
|             ps|124.58333333333333|
|        neptune|               0.0|
|          satan|0.5208333333333334|
|          saint| 6.637065637065637|
|           nmap|               0.0|
|        apache2| 38689.75465313029|
|      portsweep|               0.0|
|           back| 52894.35051546392|
|      sqlattack|             398.0|
|         xsnoop|            775.75|
|   guess_passwd| 57.58841778697001|
|         normal| 2501.925664398511|
|        rootkit|           54713.2|
|           perl|             258.0|
|     httptunnel|506.83653846153845|
|buffer_overflow|            1867.0|
|       udpstorm|               0.0|
|       multihop|1767.8333333333333|
|        ipsweep|14.666666666666666|
|           worm|            4209.0|
+---------------+------------------+
only showing top 20 rows


De igual modo, se realiza otra transformacíon agrupando con .groupBy() y através de agg  y la función avg se obtiene el promedio. Finalmente, a traves de .alias se le agrega un nombre  informativo a la columna obtenida.